In [ ]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
import polars
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
#plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import xgboost as xgb
import skforecast
import sklearn
from xgboost import XGBRegressor
from sklearn.feature_selection import RFECV
from skforecast.ForecasterAutoreg import ForecasterAutoreg
from skforecast.model_selection import bayesian_search_forecaster
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import select_features
import shap

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

from sklearn.preprocessing import LabelEncoder

Import dataframes

In [ ]:
xgdf = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/Aggregated_data_20241106.parquet', engine='pyarrow')
# xgdf_train = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/train_df_20241115.parquet', engine='pyarrow')
# xgdf_test = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/test_df_20241115.parquet', engine='pyarrow')
# xgdf_val = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/val_df_20241115.parquet', engine='pyarrow')

# xgdf = pd.concat([xgdf_train, xgdf_test, xgdf_val], axis=0, ignore_index=True)

xgdf.info()
xgdf.head()

Create extra features + One Hot Encoding for store type and item family

In [ ]:
#Data transformations
xgdf['week_number'] = xgdf['date'].dt.isocalendar().week
xgdf['month'] = xgdf['date'].dt.month
xgdf['year'] = xgdf['date'].dt.year

xgdf['unique_id'] = xgdf['store_nbr'].astype(str) + "_" + xgdf['item_nbr'].astype(str)

# Encode `store_id`
le = LabelEncoder()
xgdf['unique_id'] = le.fit_transform(xgdf['unique_id'])

xgdf = pd.get_dummies(xgdf, columns=['store_type'])
xgdf = pd.get_dummies(xgdf, columns=['item_family'])

xgdf = xgdf.loc[:, ~(xgdf == False).all()]

xgdf

Create 52 lags

In [ ]:
#Sort timeseries by unique id and date
xgdf=xgdf.sort_values(by=['unique_id','date'])

for i in range(1, 53):
    # Create the new column dynamically
    col_name = f'last{7 * i}Value'
    xgdf[col_name] = xgdf.groupby('unique_id')['unit_sales'].shift(i)

xgdf.head()

Normalize

In [ ]:
def normalize(series):
    # Ensure the series is numeric (convert to float)
    series = series.astype(float)  # Convert to float directly
    min_val = series.min()
    max_val = series.max()
    return (series - min_val) / (max_val - min_val)

col_to_normalize = ['week_number', 'year', 'month']

n_xgdf = xgdf.copy() 

for column in col_to_normalize:
    n_xgdf[column] = normalize(n_xgdf[column])

Train test validation split

In [ ]:
# Ensure 'date' is of datetime type for comparison
n_xgdf['date'] = pd.to_datetime(n_xgdf['date'])

df = n_xgdf

# 1. Filter data based on the 'date' range
# df_train = df[(df['week_number_cum'] > 138) & (df['week_number_cum'] <= 190)]
df_train = df['week_number_cum'] <= 190
df_test = df[(df['week_number_cum'] > 190) & (df['week_number_cum'] <= 216)]
df_validate = df[df['week_number_cum'] > 216]

# 2. Sort by 'date'
df_train = df_train.sort_values(by='week_number_cum')
df_test = df_test.sort_values(by='week_number_cum')
df_validate = df_validate.sort_values(by='week_number_cum')

# 3. Drop 'unit_sales' and 'date' columns for X_train and X_test
X_train = df_train.drop(columns=['unit_sales', 'date', 'last7Value'])
X_test = df_test.drop(columns=['unit_sales', 'date', 'last7Value'])

# 4. Rename 'unit_sales' to 'Label' for y_train and y_test
y_train = df_train[['unit_sales']].rename(columns={'unit_sales': 'Label'})
y_test = df_test[['unit_sales']].rename(columns={'unit_sales': 'Label'})



Train XGB model

In [ ]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred[y_pred < 0] = 0

a = y_test.rename(columns={'Label': 'y'})
b=pd.DataFrame(y_pred, columns=['y_p'])

Create dataframe which contains predictions and metrics (bias and accuracy)

In [ ]:
# Create a copy of the test dataset to avoid modifying the original dataset
df_pred = df_test.copy()

# Retain only the specified columns in the new DataFrame
df_pred = df_pred[['store_nbr', 'item_nbr', 'unit_sales', 'week_number_cum', 'date', 'unique_id', 'last14Value', 'store_cluster', 'item_class', 'perishable', 'store_type_A', 'store_type_B', 'store_type_C', 'store_type_D', 'item_family_BREAD/BAKERY', 'item_family_DAIRY', 'item_family_GROCERY I', 'item_family_POULTRY']]

# Assuming the one-hot encoded columns are ['store_type_A', 'store_type_B', 'store_type_C', ...]
# Use idxmax to find the original column (category) for each row
df_pred['store_type'] = df_pred.filter(like='store_type_').idxmax(axis=1)

# Remove the prefix 'store_type_' from the column values to get the original category
df_pred['store_type'] = df_pred['store_type'].str.replace('store_type_', '')

# Drop the one-hot encoded columns if you no longer need them
df_pred = df_pred.drop(columns=xgdf.filter(like='store_type_').columns)

# Assuming the one-hot encoded columns are ['store_type_A', 'store_type_B', 'store_type_C', ...]
# Use idxmax to find the original column (category) for each row
df_pred['item_family'] = df_pred.filter(like='item_family_').idxmax(axis=1)

# Remove the prefix 'store_type_' from the column values to get the original category
df_pred['item_family'] = df_pred['item_family'].str.replace('item_family_', '')

# Drop the one-hot encoded columns if you no longer need them
df_pred = df_pred.drop(columns=xgdf.filter(like='item_family_').columns)

# Add a new column 'y_xgb' to store the predictions from the XGBoost model
df_pred['y_xgb'] = y_pred

# Rename the 'last14Value' column to 'y_naive' to represent the naive forecast (e.g., last observed value)
df_pred = df_pred.rename(columns={'last14Value': 'y_naive'})

# Calculate the bias (error) for XGBoost predictions
# Bias is the difference between actual sales and predicted sales
df_pred['bias_xgb'] = df_pred['y_xgb'] - df_pred['unit_sales']

# Calculate the bias (error) for naive predictions
df_pred['bias_naive'] = df_pred['y_naive'] - df_pred['unit_sales']

# Calculate accuracy for XGBoost predictions
# If actual sales are 0 and predictions are non-zero, accuracy is undefined (set to NaN)
# # Otherwise, calculate accuracy as a percentage using the formula: (1 - abs(bias) / actual sales) * 100
df_pred['acc_xgb'] = np.where(
    (df_pred['unit_sales'] == 0) & (df_pred['y_xgb'] != 0),
    np.nan,
    (1 - np.abs(df_pred['bias_xgb']) / df_pred['unit_sales']) * 100
)


# Calculate accuracy for naive predictions using the same logic as above
df_pred['acc_naive'] = np.where(
    (df_pred['unit_sales'] == 0) & (df_pred['y_naive'] != 0),
    np.nan,
    (1 - np.abs(df_pred['bias_naive']) / df_pred['unit_sales']) * 100
)

# Sort the DataFrame by 'unique_id' and 'date' for better organization
df_pred = df_pred.sort_values(by=['unique_id', 'date'])

# Display the resulting DataFrame
df_pred

In [ ]:
df = df_pred

# Count how many times positive bias occurs for 'bias_xgb' and 'bias_naive'
positive_bias = np.sum(df['bias_xgb'] > 0)  # Count positive values in 'bias_xgb'
negative_bias = np.sum(df['bias_xgb'] < 0)  # Count negative values in 'bias_xgb'

positive_bias_n = np.sum(df['bias_naive'] > 0)  # Count positive values in 'bias_naive'
negative_bias_n = np.sum(df['bias_naive'] < 0)  # Count negative values in 'bias_naive'

# Output the results
print(f"Positive Bias XGB: {positive_bias}")
print(f"Negative Bias XGB: {negative_bias}")
print(f"Positive Bias Naive: {positive_bias_n}")
print(f"Negative Bias Naive: {negative_bias_n}")

positive_bias_mean = np.mean(df['bias_xgb'][df['bias_xgb'] > 0])
negative_bias_mean = np.mean(df['bias_xgb'][df['bias_xgb'] < 0])
positive_bias_n_mean = np.mean(df['bias_naive'][df['bias_naive'] > 0])
negative_bias_n_mean = np.mean(df['bias_naive'][df['bias_naive'] < 0])

# Print the results rounded to 2 decimal places
print(f"Positive Bias XGB Mean: {positive_bias_mean: .2f}")
print(f"Negative Bias XGB Mean: {negative_bias_mean: .2f}")
print(f"Positive Bias Naive Mean: {positive_bias_n_mean: .2f}")
print(f"Negative Bias Naive Mean: {negative_bias_n_mean: .2f}")

overprediction_xgb = positive_bias * positive_bias_mean * 2/3
underprediction_xgb = negative_bias * negative_bias_mean * -1/3
overprediction_naive = positive_bias_n * positive_bias_n_mean * 2/3
underprediction_naive = negative_bias_n * negative_bias_n_mean * -1/3

print(f"Overprediction XGB: {overprediction_xgb: .2f}")
print(f"Underprediction XGB: {underprediction_xgb: .2f}")
print(f"Overprediction Naive: {overprediction_naive: .2f}")
print(f"Underprediction Naive: {underprediction_naive: .2f}")

cost_xgb = overprediction_xgb + underprediction_xgb
cost_naive = overprediction_naive + underprediction_naive

difference = cost_naive - cost_xgb

print(f"Cost XGB: {cost_xgb: .2f}")
print(f"Cost Naive: {cost_naive: .2f}")
print(f"Difference (Naive - XGB): {difference: .2f}")



In [ ]:
def calculate_statistics(df):
    # Calculate means for unit sales and predictions
    mean_a = np.mean(df['unit_sales'])
    mean_p = np.mean(df['y_xgb'])
    mean_n = np.mean(df['y_naive'])
    
    # Calculate mean accuracy for XGB and Naive models
    mean_accuracy = (1 - (np.abs(mean_a - mean_p) / mean_a)) * 100
    accuracy = np.mean(df['acc_xgb'])
    
    mean_accuracy_n = (1 - (np.abs(mean_a - mean_n) / mean_a)) * 100
    accuracy_n = np.mean(df['acc_naive'])
    
    # Calculate standard deviations of accuracy
    sd_acc = np.std(df['acc_xgb'])
    sd_acc_n = np.std(df['acc_naive'])
    
    # Calculate bias and positive/negative bias for XGB and Naive models
    bias = np.mean(df['bias_xgb'])
    positive_bias = np.mean(df['bias_xgb'][df['bias_xgb'] > 0])
    negative_bias = np.mean(df['bias_xgb'][df['bias_xgb'] < 0])
    
    bias_n = np.mean(df['bias_naive'])
    positive_bias_n = np.mean(df['bias_naive'][df['bias_naive'] > 0])
    negative_bias_n = np.mean(df['bias_naive'][df['bias_naive'] < 0])
    
    # Calculate standard deviations of bias
    sd_bias = np.std(df['bias_xgb'])
    sd_bias_n = np.std(df['bias_naive'])
    
    # Store results in a dictionary
    results = {
        'Metric': [
            'Mean Accuracy', 'Accuracy', 'Mean Accuracy Naive', 'Accuracy Naive',
            'Standard Deviation of Accuracy', 'Standard Deviation of Accuracy Naive',
            'Bias', 'Positive Bias', 'Negative Bias', 
            'Bias Naive', 'Positive Bias Naive', 'Negative Bias Naive',
            'Standard Deviation of Bias', 'Standard Deviation of Bias Naive'
        ],
        'Value': [
            mean_accuracy, accuracy, mean_accuracy_n, accuracy_n,
            sd_acc, sd_acc_n,
            bias, positive_bias, negative_bias,
            bias_n, positive_bias_n, negative_bias_n,
            sd_bias, sd_bias_n
        ]
    }
    
    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

In [ ]:
def calculate_metrics_grouped(df, column):
    store_stats = []
    
    # Iterate over each group's data
    for store, group in df.groupby(column):
        # # Skip calculation if the group value is 0 or missing (NaN)
        # if store == 0 or pd.isna(store):
        #     continue
        
        # Calculate statistics for the current group using the existing function
        df_metrics = calculate_statistics(group)
        
        # Add the column value (e.g., store number) to each metric's result for reference
        df_metrics[column] = store
        
        # Append each group's result to the list
        store_stats.append(df_metrics)
    
    # Concatenate all group statistics into a single DataFrame
    store_means = pd.concat(store_stats).reset_index(drop=True)

    # Pivot the DataFrame to make column values into columns and Metric rows
    store_metrics_pivot = store_means.pivot(index='Metric', columns=column, values='Value')

    # Drop columns with all NA values
    store_metrics_pivot = store_metrics_pivot.dropna(axis=1, how='all')

    return store_metrics_pivot


In [ ]:
def plot_bias_comparison(df):
    """
    Plots a comparison of Negative Bias, Negative Bias Naive, Positive Bias, and Positive Bias Naive
    across stores from a given DataFrame, sorted by Positive Bias Naive - Positive Bias.

    Parameters:
    df (pd.DataFrame): DataFrame where rows represent metrics ('Negative Bias', 'Negative Bias Naive',
                        'Positive Bias', 'Positive Bias Naive') and columns are store numbers.
    """
    # Extract the data to plot
    stores = list(df.columns)  # Get store numbers as list of columns excluding the 'Metric' column
    negative_bias = df.loc['Negative Bias', stores].values.astype(float)  # Negative Bias values
    negative_bias_naive = df.loc['Negative Bias Naive', stores].values.astype(float)  # Negative Bias Naive values
    positive_bias = df.loc['Positive Bias', stores].values.astype(float)  # Positive Bias values
    positive_bias_naive = df.loc['Positive Bias Naive', stores].values.astype(float)  # Positive Bias Naive values

    # Calculate the difference for sorting: Positive Bias Naive - Positive Bias
    bias_diff = positive_bias_naive - positive_bias

    # Sort the stores based on the difference in descending order
    sorted_indices = np.argsort(bias_diff)[::-1]  # Sort indices in descending order
    sorted_stores = np.array(stores)[sorted_indices]  # Reorder stores based on sorted indices

    # Reorder all other data according to sorted stores
    negative_bias = negative_bias[sorted_indices]
    negative_bias_naive = negative_bias_naive[sorted_indices]
    positive_bias = positive_bias[sorted_indices]
    positive_bias_naive = positive_bias_naive[sorted_indices]

    # Set up the plot
    x = np.arange(len(sorted_stores))  # Store positions for the x-axis
    width = 0.2  # Width of the bars

    fig, ax = plt.subplots(figsize=(15, 8))

    # Plot the bars for each category
    ax.bar(x - width, negative_bias, width, label='Negative Bias')
    ax.bar(x, negative_bias_naive, width, label='Negative Bias Naive')
    ax.bar(x + width, positive_bias, width, label='Positive Bias')
    ax.bar(x + 2 * width, positive_bias_naive, width, label='Positive Bias Naive')

    # Add labels and formatting
    ax.set_xlabel('Stores')
    ax.set_ylabel('Bias Value')
    ax.set_title('Comparison of Negative and Positive Bias Across Stores')
    ax.set_xticks(x)
    ax.set_xticklabels(sorted_stores, rotation=90)
    ax.legend()

    # Display the plot
    plt.tight_layout()
    plt.show()


In [ ]:
df_metrics = calculate_statistics(df_pred)
df_metrics

In [ ]:
df_metrics_stores = calculate_metrics_grouped(df_pred, 'store_nbr')
df_metrics_stores

df_metrics_item_family = calculate_metrics_grouped(df_pred, 'item_family')
df_metrics_item_family

df_metrics_item_class = calculate_metrics_grouped(df_pred, 'item_class')
df_metrics_item_class

df_metrics_store_type = calculate_metrics_grouped(df_pred, 'store_type')
df_metrics_store_type

df_metrics_store_cluster = calculate_metrics_grouped(df_pred, 'store_cluster')
df_metrics_store_cluster

df_metrics_perishable = calculate_metrics_grouped(df_pred, 'perishable')
df_metrics_perishable

df_metrics_weeknbr = calculate_metrics_grouped(df_pred, 'week_number_cum')
df_metrics_weeknbr

In [ ]:
plot_stores = plot_bias_comparison(df_metrics_stores)
plot_stores

plot_item_family = plot_bias_comparison(df_metrics_item_family)
plot_item_family

plot_item_class = plot_bias_comparison(df_metrics_item_class)
plot_item_class

plot_store_type = plot_bias_comparison(df_metrics_store_type)
plot_store_type

plot_perishable = plot_bias_comparison(df_metrics_perishable)
plot_perishable

plot_store_cluster = plot_bias_comparison(df_metrics_store_cluster)
plot_store_cluster

plot_week_nbr = plot_bias_comparison(df_metrics_weeknbr)
plot_week_nbr


In [ ]:
#grouped_df = df_pred.groupby('week_number_cum')[['y_pred', 'unit_sales']].sum().reset_index()
grouped_df = df_pred[(df_pred['item_nbr'] == 103520) & (df_pred['store_nbr'] == 1)]

# Step 2: Plot the results
plt.figure(figsize=(12, 6))
plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='Predicted Sales (y_pred)', color='blue')
plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='orange')
plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction', color='red')

# Add titles and labels
plt.title('Sum of y_pred and Unit Sales Over Weeks')
plt.xlabel('Week Number Cumulative')
plt.ylabel('Sales')
plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
plt.legend()
plt.grid()

# Show the plot
plt.show()

In [ ]:
#grouped_df = df_pred.groupby('week_number_cum')[['y_pred', 'unit_sales']].sum().reset_index()
grouped_df = df_pred[(df_pred['item_nbr'] == 103520) & (df_pred['store_nbr'] == 1)]

# Step 2: Plot the results
plt.figure(figsize=(12, 6))
plt.plot(grouped_df['week_number_cum'], grouped_df['bias_xgb'], marker='o', label='Predicted Sales (y_pred)', color='blue')
# plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='orange')
plt.plot(grouped_df['week_number_cum'], grouped_df['bias_naive'], marker='o', label='Naive prediction', color='red')

# Add titles and labels
plt.title('Sum of y_pred and Unit Sales Over Weeks')
plt.xlabel('Week Number Cumulative')
plt.ylabel('Sales')
plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
plt.legend()
plt.grid()

# Show the plot
plt.show()

In [ ]:
df_metrics_weeknbr


# Bias	0.991012	-3.575474	-3.527770	0.899179	1.951780	-2.717944	-3.790900	1.394907	2.005176	2.692322	...	-4.082209	-8.913561	3.735825	-4.302500	1.018056	4.094487	4.046926	-0.590725	3.588645	4.506714
# Bias Naive	-0.971684	-4.693805	-5.063268	4.608160	3.840239	-4.124322	-5.488238	3.262603	5.256981	2.840175	...	-3.488959	-7.317317	9.907116	2.998332	-0.146018	11.036333	1.423326	-4.627019	-1.503645	4.828174
# Mean Accuracy	96.980110	90.917806	90.686879	97.413157	94.266098	93.010150	90.409477	96.084094	94.148999	91.787029	...	91.255769	82.413658	89.842080	90.977471	97.242808	88.828113	88.600259	98.568872	90.301962	87.635476
# Mean Accuracy Naive	97.039034	88.077090	86.633234	86.742851	88.718228	89.393310	86.115417	90.840940	84.660375	91.335987	...	92.526532	85.563023	73.061985	93.712384	99.604526	69.887111	95.990630	88.790294	95.936496	86.753513
# Negative Bias	-9.820827	-13.153575	-14.276734	-11.145329	-10.392314	-13.380919	-14.605677	-11.110574	-10.745089	-11.800099	...	-24.417402	-32.528988	-15.390956	-20.836000	-16.917084	-14.071371	-13.737403	-15.397860	-13.378547	-14.244827
# Negative Bias Naive	-11.580584	-15.697789	-16.538763	-11.972717	-11.223405	-15.148617	-16.373854	-11.556795	-10.353929	-13.076480	...	-32.530296	-37.303066	-14.165239	-29.128222	-21.381594	-13.001633	-14.451592	-18.290194	-14.441039	-15.815772
# Positive Bias	9.220966	8.484726	8.858513	10.203545	11.585097	10.359572	9.144983	10.266397	9.876227	11.899906	...	17.917753	17.389734	19.374748	19.952103	19.202095	22.096117	21.844492	19.218687	18.022263	19.143244
# Positive Bias Naive

In [ ]:
def plot_bias_comparison2(df):
    """
    Plots a comparison of Bias and Bias Naive values across cumulative weeks from a given DataFrame.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing 'Bias' and 'Bias Naive' columns along with 'week_number_cum'.
    """
    # Extract columns for the plot
    weeks = df.columns  # Assuming columns are 'week_number_cum'
    bias = df.loc['Bias', weeks].values  # Bias values from the 'Bias' row
    bias_naive = df.loc['Bias Naive', weeks].values  # Bias Naive values from the 'Bias Naive' row
    
    # Step 2: Plot the results
    plt.figure(figsize=(12, 6))
    
    # Plot Bias and Bias Naive
    plt.plot(weeks, bias, marker='o', label='Bias', color='blue')
    plt.plot(weeks, bias_naive, marker='o', label='Bias Naive', color='red')

    # Add titles and labels
    plt.title('Comparison of Bias vs Bias Naive Over Weeks')
    plt.xlabel('Week Number Cumulative')
    plt.ylabel('Bias Value')
    plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
    plt.legend()
    plt.grid(True)

    # Show the plot
    plt.tight_layout()
    plt.show()

# Assuming df_metrics_weeknbr is your dataframe
plot_bias_comparison2(df_metrics_weeknbr)
